# Common Table Expressions (the WITH clause)

As queries grow, they can become hard to read, especially when you need an
intermediate result. A **Common Table Expression**, or CTE, lets you name a
query and then use it like a temporary table. You write it with the **WITH**
keyword. CTEs make complex queries readable by breaking them into named
steps.

## The WITH clause

The shape of a CTE is:

```sql
WITH cte_name AS (
    SELECT ...
)
SELECT ... FROM cte_name ...;
```

You define a named subquery once, then refer to it by name in the main
query. Let's start simple on Chinook: name the set of long tracks, then
count them.

In [ ]:
from rel_db_learning import get_chinook_connection, get_project_connection, run_sql
from rel_db_learning.widgets.self_assessment import sql_query_check

chinook = get_chinook_connection()
project = get_project_connection()

run_sql("""
    WITH long_tracks AS (
        SELECT name, milliseconds
        FROM track
        WHERE milliseconds > 300000
    )
    SELECT COUNT(*) AS long_track_count
    FROM long_tracks
""", chinook)

The real value appears with aggregates. Suppose we want artists who have more
than 10 albums. We first build a CTE that counts albums per artist, then
filter it:

In [ ]:
run_sql("""
    WITH album_counts AS (
        SELECT artist_id, COUNT(*) AS album_count
        FROM album
        GROUP BY artist_id
    )
    SELECT artist_id, album_count
    FROM album_counts
    WHERE album_count > 10
    ORDER BY album_count DESC, artist_id
""", chinook)

Without the WITH clause, you would need a HAVING clause or a subquery. The CTE makes
the two steps (count, then filter) clear and separate. You can even reference
a CTE more than once in the main query.

## CTEs on your Library

Let's find members and how many loans they have, using a CTE to compute the
counts first:

In [ ]:
run_sql("""
    WITH loan_counts AS (
        SELECT member_id, COUNT(*) AS loan_count
        FROM loan
        GROUP BY member_id
    )
    SELECT m.full_name, lc.loan_count
    FROM member m
    JOIN loan_counts lc ON m.member_id = lc.member_id
    ORDER BY m.full_name
    LIMIT 10
""", project)

Here the CTE computed loan counts per member, then the main query joined
those counts to member names. Reading it top to bottom matches how you think
about the problem: first count, then attach names.

## Exercises

**Exercise 1:** Using a CTE that counts loans per member, return the
`full_name` and loan count of every member who has exactly 3 loans. Join the
CTE to the member table.

In [ ]:
my_sql_1 = "???"

sql_query_check(
    student_sql=my_sql_1,
    correct_sql="m04_nb_09_exercise_1",
    connection=project,
)

**Exercise 2:** Using a CTE that counts loans per book, return the `title`
and count of every book loaned more than once. Join the CTE to the book
table.

In [ ]:
my_sql_2 = "???"

sql_query_check(
    student_sql=my_sql_2,
    correct_sql="m04_nb_09_exercise_2",
    connection=project,
)

**Exercise 3:** Using a CTE that counts members per country, return each
`country` and its member count for countries with more than 3 members.

In [ ]:
my_sql_3 = "???"

sql_query_check(
    student_sql=my_sql_3,
    correct_sql="m04_nb_09_exercise_3",
    connection=project,
)

## The same in Python with Ibis

In Ibis, you build intermediate results by assigning expressions to
variables, which plays the same role as a CTE: name a step, then build on
it. Here we count albums per artist, then keep the busy ones.

In [ ]:
import ibis
from rel_db_learning import get_ibis_chinook

ibis_chinook = get_ibis_chinook()
album = ibis_chinook.table("album")

album_counts = album.group_by("artist_id").aggregate(album_count=album.count())
result = album_counts.filter(album_counts.album_count > 10)
result.execute()

And the SQL Ibis generated. Notice Ibis often expresses this as a subquery or
a WITH clause, the same idea you just learned:

In [ ]:
print(ibis.to_sql(result))

### Optional: try it on your Library

Build an intermediate count on your Library with Ibis. The cell below counts
loans per member. Try filtering it to members with 3 loans. This is optional
and not checked.

In [ ]:
from rel_db_learning import get_ibis_project

ibis_project = get_ibis_project()
loan = ibis_project.table("loan")

loan.group_by("member_id").aggregate(loan_count=loan.count()).execute()

## Experiment freely

Try writing a query with more than one CTE, or referencing a CTE twice. CTEs
are a clean way to break a hard query into readable steps.

In [ ]:
run_sql("""
    WITH book_loans AS (
        SELECT book_id, COUNT(*) AS cnt FROM loan GROUP BY book_id
    )
    SELECT b.title, bl.cnt
    FROM book b
    JOIN book_loans bl ON b.book_id = bl.book_id
    ORDER BY bl.cnt DESC, b.title
    LIMIT 10
""", project)

## What's next

You can now structure complex queries with CTEs. Next you will learn about
**views**, saved queries that behave like virtual tables, and you will grow
your Library again with new columns, a new table, and several views.